# 03 — Policy Rules: a Deterministic Baseline for Expense Auditing

This notebook runs the **rules engine** that audits every generated expense claim
against the company travel & expense policy (`policy.yaml`). The engine itself now
lives in the `expense_agent` package (`src/expense_agent/rules.py` and `models.py`,
with its test suite in `tests/`); the notebook loads the data, imports the engine,
runs it end to end, and scores the results. It is the deterministic baseline that
the LLM agent (notebook 05) will later be measured against (notebook 06).

**What it does, end to end:**

| Section | What happens | Policy sections |
|---|---|---|
| 1 | Load policy + claims (no labels); import the rules engine from `expense_agent.rules` | — |
| 2 | Validate all claims through the typed `Claim` model (`expense_agent.models`) | — |
| 3–8 | *Moved to the package:* single-claim rules, category caps, fare/rental/vendor/mileage rules, review flags, escalation flags, cross-claim duplicates & splits — see `src/expense_agent/rules.py` and `tests/test_rules.py` | POL-1.x, POL-2.x, POL-3.x, POL-5.1/5.2 |
| 9 | `run_rules()`: one decision row per claim | POL-4, POL-5 |
| 10/10b | Score rule decisions against ground-truth labels; diagnose missed reviews | evaluation only |

**Design principles used throughout:**

- **Labels never feed a rule.** Ground-truth labels are loaded only in section 10,
  strictly for evaluation. Section 1 even asserts the claims frame carries no label column.
- **`policy.yaml` is the single source of truth.** Thresholds, caps, vendor lists, and
  violation codes all come from the loaded policy dict — never hard-coded — and a drift
  check (`tests/test_models.py::test_violation_codes_match_policy`) fails if the code's
  `ViolationCode` literal and the config disagree.
- **Violations vs flags.** A `Violation` is a broken rule that *denies* a claim; a `Flag`
  only routes it to *review*. Decision precedence is deny > review > approve.
- **Every rule cites its policy section** via a validated `policy_ref` (e.g. `POL-2.2`),
  so each finding is auditable back to the written policy.
- **Boundary and anchor tests live in `tests/`** (run `uv run pytest` from the project
  root). They pin exact thresholds ($75.00 vs $75.01, 60 vs 61 days) plus *anchor*
  assertions against known counts in the generated dataset, so silent regressions
  fail loudly. The notebook keeps only data-level and pipeline-level invariant checks.

## 1: Imports, policy, claims

Load the policy config and the 330 generated claims. Labels are loaded here only so
later cells can reuse the loader, but they are **not** merged into the claims — the
final assert proves `in_policy` never leaks into `claims_df`, keeping rule-building
honest. `%autoreload` lets edits in the `expense_agent` package take effect without
restarting the kernel.

In [1]:
# 1. Imports, policy, claims (no labels)
%load_ext autoreload
%autoreload 2

import re

import pandas as pd

from expense_agent.data import load_claims, load_labels
from expense_agent.paths import GENERATED_DIR, POLICY_PATH
from expense_agent.policy import load_policy

policy = load_policy(POLICY_PATH)
claims_df = load_claims(GENERATED_DIR)
labels_df = load_labels(GENERATED_DIR)

print(f"policy v{policy['version']} claims={len(claims_df)}")
assert len(claims_df) == 330, "Expected the generated set (330 claims)"
assert "in_policy" not in claims_df.columns, "Label leak in claims_df"

policy v1.0 claims=330


In [2]:
# 1b Rules engine: imported from the package (replaces cells 3b-8c)
from expense_agent.models import to_claims
from expense_agent.rules import (
    collect_findings,
    rule_decision,
    run_rules,
)

# check: names point to the package, not a leftover notebook copy
for func in (run_rules, rule_decision, collect_findings):
    assert func.__module__ == "expense_agent.rules", func.__module__
print("all rules come from expense_agent.rules")

all rules come from expense_agent.rules


## 2: Typed Input

Pydantic models (defined in `expense_agent.models`) form the contract for everything
downstream:

- **`Claim`** — one expense line item, `frozen=True` (immutable, so rules can't mutate
  inputs) and `extra="ignore"` (tolerates surplus DataFrame columns). The
  `blank_to_none` validator runs on every field *before* type coercion, converting
  pandas `NaN`/`<NA>` and empty/whitespace strings to `None` so optional fields behave.
- **`Violation`** — one broken rule on one claim. `extra="forbid"` plus a regex-validated
  `policy_ref` (`POL-d.d`) and non-empty `detail` make malformed findings impossible
  to construct.
- **`to_claims`** validates every row and fails loudly on bad data rather than skipping it.

The cell below checks only the data: all 330 rows validate with unique IDs. The model
contract tests — the `ViolationCode` literal matching `policy.yaml`'s `violation_codes`
(single source of truth), rejection of a miscased policy reference (`Pol-2.2`), and
frozen-model immutability — now live in `tests/test_models.py`.

In [3]:
# 2 - checks
claims = to_claims(claims_df)
assert len(claims) == 330
assert len({c.claim_id for c in claims}) == 330, "claim_id not unique"

### Cells 3–8c: moved to the package (Session 11, 2026-11-26)
Rule functions now live in `src/expense_agent/rules.py` and models in
`src/expense_agent/models.py`; tests in `tests/test_rules.py` and
`tests/test_models.py`. Imported in cell 1b.

## 9: run_rules()

Runs the pipeline imported from `expense_agent.rules` (cell 1b):

- **`collect_findings`** runs all rule groups in order — single-claim rules, then
  duplicates, then splits (skipping duplicate IDs), then flag + escalation rules —
  returning violations and flags separately.
- **`rule_decision`** encodes the precedence from POL-4/POL-5: any violation ⇒
  **deny**; else any flag ⇒ **review**; else **approve**.
- **`run_rules`** groups findings by claim and emits exactly one row per claim with
  the decision, violation codes, flag codes, cited policy sections, and human-readable
  details — the full audit trail for each claim.

Invariant checks: one row per claim, unique IDs, and every denied row carries at least
one violation (a deny can never be unexplained).

In [4]:
rule_results = run_rules(claims, policy)

assert len(rule_results) == len(claims), "one row per claim"
assert rule_results["claim_id"].is_unique
denied = rule_results["rule_decision"] == "deny"
assert rule_results.loc[denied, "violations"].map(len).gt(0).all()

software = next(c for c in claims if c.category == "Software")
odd_vendor = software.model_copy(update={"merchant": "Unlisted SaaS Inc"})
row = run_rules([odd_vendor], policy).iloc[0]
assert row["rule_decision"] == "review", row["violations"]
assert row["flags"] == ["unapproved_vendor"]

print(rule_results["rule_decision"].value_counts().to_string())
print(rule_results[denied].head(5).to_string())

rule_decision
approve    237
deny        62
review      31
        claim_id rule_decision           violations         flags         policy_refs                                                                                                     details
9   R-E001-05-L2          deny           [over_cap]  [high_value]  [POL-2.9, POL-5.1]                                          [Home Office $1,196.21 over cap $500.00, $1,196.21 over $1,000.00]
16  R-E001-07-L3          deny      [premium_class]  [high_value]  [POL-2.1, POL-5.1]  [Business on 3.9h flight, preapproved=True (needs Business, >= 6h, preapproved), $1,927.76 over $1,000.00]
25  R-E001-10-L2          deny    [missing_receipt]            []           [POL-1.3]                                                                            [$232.09 over $75.00 no receipt]
26  R-E002-01-L1          deny  [split_transaction]            []           [POL-1.5]                                                   [split with R-E002-01-L2: tot

## 10: Score vs Labels

The only place ground-truth labels appear — strictly for **evaluation**; nothing here
feeds back into a rule.

Setup: labels and amounts are merged onto `rule_results` with `validate="one_to_one"`
guards. One scoring adjustment is needed first: labels mark only one half of each
split purchase as `split_transaction`, but POL-1.5 correctly denies *both* halves.
The `same_purchase` mask finds in-policy claims whose **only** violation is
`split_transaction` and whose detail names a labeled split partner (via the
`CLAIM_ID` regex) — those are the same purchase, not false positives.

Four read-outs follow:

1. **Catch rate & false positives** — how many out-of-policy claims the rules deny,
   whether the labeled violation type appears among the rule's codes, and how many
   in-policy claims were wrongly denied (after the split adjustment).
2. **Confusion matrix** — labeled decision (rows) vs rule decision (columns).
3. **Rules-only baseline for notebook 06** — false-approval rate, dollars leaked
   through false approvals, and label-deny claims parked in review (dollars at risk).
4. **What rules miss** — out-of-policy claims the rules did *not* deny, broken down
   by violation type: exactly the gap the LLM agent must close in notebook 05.

Anchors lock in the headline result: 59 out-of-policy claims caught, 3 split halves
correctly denied with their purchase, and **zero** false positives.

Cell 10b then drills into the review → approve gap: for every labeled-review claim it
cross-tabs the review reason against the rule decision and lists the misses with the
claim text the rules would have had to judge — the semantic cases rules can't see.

In [5]:
# 10 - Score rules vs labels (evaluation only: labels never feed a rule)
CLAIM_ID = re.compile(r"R-E\d{3}-\d{2}-L\d+")


def named_claims(details: list[str]) -> set[str]:
    """Return every claim ID mentioned in a row's finding details."""
    return {cid for detail in details for cid in CLAIM_ID.findall(detail)}


labels = load_labels(GENERATED_DIR)

assert set(labels["claim_id"]) == set(rule_results["claim_id"]), "ID mismatch"
assert labels["in_policy"].dtype == bool, "in_policy must load as bool"

scored = rule_results.merge(
    labels, on="claim_id", validate="one_to_one"
).merge(
    claims_df[["claim_id", "amount"]], on="claim_id", validate="one_to_one"
)

out_of_policy = ~scored["in_policy"]
rule_deny = scored["rule_decision"] == "deny"
label_deny = scored["expected_decision"] == "deny"

# Splits are scored per purchase (POL-1.5): a clean half denied only
# because its labeled partner is a split is the same purchase, not a FP.
labeled_splits = set(
    scored.loc[scored["violation_type"] == "split_transaction", "claim_id"]
)
split_only = scored["violations"].map(
    lambda codes: codes == ["split_transaction"]
)
same_purchase = (
    scored["in_policy"]
    & split_only
    & scored["details"].map(lambda ds: bool(named_claims(ds) & labeled_splits))
)

# 1. Catch rate and false positives
caught = out_of_policy & rule_deny
false_pos = scored["in_policy"] & rule_deny & ~same_purchase
type_match = pd.Series(
    [
        label in codes
        for label, codes in zip(
            scored["violation_type"], scored["violations"], strict=True
        )
    ],
    index=scored.index,
)
print(f"caught: {caught.sum()} of {out_of_policy.sum()} out-of-policy")
print(f"  labeled type among rule codes: {(caught & type_match).sum()}")
print(f"split halves denied with their purchase: {same_purchase.sum()}")
print(f"false positives (in policy, denied): {false_pos.sum()}")

# 2. Decision confusion matrix: rows = label, columns = rules
print()
print(
    pd.crosstab(
        scored["expected_decision"],
        scored["rule_decision"],
        margins=True,
    )
)

# 3. Rules-only baseline for notebook 06
false_approve = label_deny & (scored["rule_decision"] == "approve")
to_review = label_deny & (scored["rule_decision"] == "review")
fa_rate = false_approve.sum() / label_deny.sum()
leaked = scored.loc[false_approve, "amount"].sum()
at_risk = scored.loc[to_review, "amount"].sum()
print()
print(
    f"false-approval rate: {fa_rate:.1%} "
    f"({false_approve.sum()} of {label_deny.sum()})"
)
print(f"dollars leaked: ${leaked:,.2f}")
print(f"label deny sent to review: {to_review.sum()} (${at_risk:,.2f})")

# 4. What rules miss: the agent's job in notebook 05
missed = out_of_policy & ~rule_deny
print()
print(
    pd.crosstab(
        scored.loc[missed, "violation_type"],
        scored.loc[missed, "rule_decision"],
        margins=True,
    )
)

# Anchors: B + A applied (Session 8)
assert caught.sum() == 59, f"expected 59 caught, got {caught.sum()}"
assert same_purchase.sum() == 3, "one clean half per split purchase"
assert false_pos.sum() == 0, "rules denied an in-policy claim"

caught: 59 of 82 out-of-policy
  labeled type among rule codes: 57
split halves denied with their purchase: 3
false positives (in policy, denied): 0

rule_decision      approve  deny  review  All
expected_decision                            
approve                222     3       6  231
deny                     6    59      17   82
review                   9     0       8   17
All                    237    62      31  330

false-approval rate: 7.3% (6 of 82)
dollars leaked: $2,394.30
label deny sent to review: 17 ($2,900.89)

rule_decision        approve  review  All
violation_type                           
personal_extension         6       0    6
prohibited_category        0      17   17
All                        6      17   23


In [6]:
# 10b - Diagnose review -> approve: which POL-5 escalations do rules miss?
# (evaluation only: labels explain the gap, they never feed a rule).

label_review = scored["expected_decision"] == "review"
missed_review = label_review & (scored["rule_decision"] == "approve")

# Every labeled review claim: its reason vs what the rules decided
print("label review, by reason (rows) and rule decision (columns):")
print(
    pd.crosstab(
        scored.loc[label_review, "review_reason"],
        scored.loc[label_review, "rule_decision"],
        margins=True,
    )
)

# The misses, with the claim text the rules would have to judge
missed_detail = scored.loc[missed_review, ["claim_id", "review_reason"]]
missed_detail = missed_detail.merge(
    claims_df[["claim_id", "category", "business_purpose"]],
    on="claim_id",
    validate="one_to_one",
)
print()
print(missed_detail.to_string(index=False))

# Anchor: the 9 from the cell 10 confusion matrix
assert missed_review.sum() == 9, f"expected 9, got {missed_review.sum()}"

label review, by reason (rows) and rule decision (columns):
rule_decision  approve  review  All
review_reason                      
high_value           0       1    1
near_cap             0       4    4
vague_purpose        9       3   12
All                  9       8   17

    claim_id review_reason        category business_purpose
R-E001-02-L3 vague_purpose        Software       Work stuff
R-E002-08-L2 vague_purpose        Software             Team
R-E003-04-L2 vague_purpose Office Supplies         Business
R-E004-02-L1 vague_purpose         Airfare          Meeting
R-E004-04-L4 vague_purpose    Client Meals          Meeting
R-E005-08-L1 vague_purpose         Airfare       Work stuff
R-E006-09-L3 vague_purpose           Meals             Misc
R-E006-10-L3 vague_purpose           Meals       Work stuff
R-E007-02-L1 vague_purpose         Airfare          Meeting


### Cells 11a–11c: moved to tests/ (Session 11, 2026-11-26)
Fixtures in `tests/conftest.py`; rule tests in `tests/test_rules.py`.
Run from the project root: `uv run pytest`

---

## Summary

**What this stage delivers:** a fully deterministic, policy-cited rules engine — typed
`Claim`/`Violation`/`Flag` models, six single-claim rules, two review-flag rules, three
escalation flags, two cross-claim rules (duplicates, splits), and a `run_rules()`
pipeline producing one auditable decision row per claim. The engine now lives in the
`expense_agent` package (`src/expense_agent/rules.py`, `models.py`) with its boundary
and anchor tests in `tests/` (`uv run pytest`); this notebook validates the data, runs
the pipeline, and scores it against labels.

**Headline results (asserted, not just printed):**

- **59** out-of-policy claims caught by rules, with **0 false positives** — rules never
  deny an in-policy claim.
- **3** clean split halves correctly denied together with their labeled partner
  (splits are judged per purchase, per POL-1.5).
- Every denial cites the exact policy section it enforces.

**What rules can and cannot do:**

- Rules excel at *mechanical* policy: thresholds, caps, dates, exact-match duplicates,
  arithmetic splits. They are cheap, instant, explainable, and perfectly repeatable.
- Rules cannot judge *semantics*: whether a description masks a prohibited purchase,
  whether a business purpose is plausible, or whether a vendor name on review really
  needs IT approval. Those cases surface in the section-10 "missed" table and the
  cell-10b review diagnosis — this is precisely the work handed to the LLM agent in
  **notebook 05**.
- The section-10 metrics (false-approval rate, dollars leaked, dollars at risk in
  review) form the **rules-only baseline** that **notebook 06** uses to measure whether
  the agent actually adds value over plain rules.

**Patterns worth reusing:** config-driven thresholds with a code↔config drift check
(`tests/test_models.py`), frozen Pydantic models as rule contracts, violation-vs-flag
severity split with deny > review > approve precedence, notebook logic promoted to an
importable package once stable (with asserts proving imports resolve to the package),
boundary tests at every threshold edge, and anchor asserts that pin known dataset
counts so regressions fail loudly.